In [10]:
import pandas as pd

# scikit-learn에서 제공하는 유방암 데이터셋을 로드합니다.
# 유방암 진단 확률(Logistic Regression)을 구하는 아주 대표적인 데이터입니다.
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
# 전체 데이터를 DataFrame으로 변환합니다.
df = pd.DataFrame(data.data, columns=data.feature_names)
# TARGET 값을 복사합니다 (0: 악성, 1: 양성)
df['TARGET'] = data.target
# 예제 코드와의 호환성을 맞추기 위해 가상의 ID 컬럼을 추가합니다.
df['ID'] = range(len(df))

# 예제를 위해 80%는 train, 20%는 test로 인위적으로 나눕니다.
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)
# test 데이터에서는 정답지인 TARGET 컬럼을 제거합니다. (일반적인 test.csv의 구조)
test = test.drop(columns=['TARGET'])

display(train.head())
display(test.head())

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,TARGET,ID
204,12.47,18.60,81.09,481.9,0.09965,0.1058,0.08005,0.03821,0.1925,0.06373,...,96.05,677.9,0.1426,0.2378,0.2671,0.10150,0.3014,0.08750,1,204
70,18.94,21.31,123.60,1130.0,0.09009,0.1029,0.10800,0.07951,0.1582,0.05461,...,165.90,1866.0,0.1193,0.2336,0.2687,0.17890,0.2551,0.06589,0,70
131,15.46,19.48,101.70,748.9,0.10920,0.1223,0.14660,0.08087,0.1931,0.05796,...,124.90,1156.0,0.1546,0.2394,0.3791,0.15140,0.2837,0.08019,0,131
431,12.40,17.68,81.47,467.8,0.10540,0.1316,0.07741,0.02799,0.1811,0.07102,...,89.61,515.8,0.1450,0.2629,0.2403,0.07370,0.2556,0.09359,1,431
540,11.54,14.44,74.65,402.9,0.09984,0.1120,0.06737,0.02594,0.1818,0.06782,...,78.78,457.8,0.1345,0.2118,0.1797,0.06918,0.2329,0.08134,1,540


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,ID
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.18600,0.2750,0.08902,1
8,13.00,21.82,87.50,519.8,0.12730,0.19320,0.18590,0.09353,0.2350,0.07389,...,30.73,106.20,739.3,0.1703,0.5401,0.5390,0.20600,0.4378,0.10720,8
13,15.85,23.95,103.70,782.7,0.08401,0.10020,0.09938,0.05364,0.1847,0.05338,...,27.66,112.00,876.5,0.1131,0.1924,0.2322,0.11190,0.2809,0.06287,13
14,13.73,22.61,93.60,578.3,0.11310,0.22930,0.21280,0.08025,0.2069,0.07682,...,32.01,108.80,697.7,0.1651,0.7725,0.6943,0.22080,0.3596,0.14310,14
20,13.08,15.71,85.63,520.0,0.10750,0.12700,0.04568,0.03110,0.1967,0.06811,...,20.49,96.09,630.5,0.1312,0.2776,0.1890,0.07283,0.3184,0.08183,20


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# drop(): 불필요한 식별자(ID)와 예측 대상인 타겟(TARGET) 변수를 제거하여 입력 피처(X)만 남깁니다.
train_x = train.drop(columns=['TARGET', 'ID'])
test_x = test.drop(columns=['ID'])

# 타겟 변수(정답 레이블) 분리
train_y = train['TARGET']

# StandardScaler(): 각 피처의 평균을 0, 분산을 1로 맞추는 표준화(Standardization)를 수행하는 객체입니다.
# 스케일링(표준화)
scaler = StandardScaler()

# fit_transform(): 학습 데이터(train_x)의 평균과 표준편차를 계산(fit)하고, 동시에 데이터를 변환(transform)합니다.
train_x_scaled = scaler.fit_transform(train_x)

# transform(): 학습 데이터에서 계산된 기준(평균, 표준편차)을 그대로 사용하여 테스트 데이터(test_x)를 변환합니다. (데이터 누수 방지)
test_x_scaled = scaler.transform(test_x)

# 스케일링된 데이터프레임 생성
# 원본 인덱스를 유지해야 train_y와 인덱스가 맞습니다.
columns = train_x.columns
train_x = pd.DataFrame(train_x_scaled, columns=columns, index=train.index)
test_x = pd.DataFrame(test_x_scaled, columns=columns, index=test.index)

# sm.add_constant(): statsmodels 라이브러리를 사용하여 회귀 분석을 할 때, 회귀식의 y절편(상수항, Bias)을 계산하기 위해 모든 값이 1인 초기 컬럼을 추가합니다.
train_x = sm.add_constant(train_x) # 상수항 추가
test_x = sm.add_constant(test_x) # 상수항 추가

# train_test_split(): 전체 학습 데이터를 모델 학습용(train)과 검증용(validation)으로 분할합니다.
# - test_size=0.2: 검증용 데이터로 전체의 20%를 사용합니다.
# - random_state=42: 코드 실행 시마다 동일하게 분할되도록 무작위 시드를 고정합니다.
train_x, val_x, train_y, val_y = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
 )

In [14]:
model = sm.OLS(train_y, train_x)
result = model.fit()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:                 TARGET   R-squared:                       0.796
Model:                            OLS   Adj. R-squared:                  0.778
Method:                 Least Squares   F-statistic:                     43.35
Date:                Fri, 03 Apr 2026   Prob (F-statistic):           1.12e-96
Time:                        22:22:04   Log-Likelihood:                 38.956
No. Observations:                 364   AIC:                            -15.91
Df Residuals:                     333   BIC:                             104.9
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

## 회귀 모델이란 무엇인가: 가장 기초부터 이해하기

회귀 모델은 **입력 변수들로부터 어떤 숫자 값을 예측하는 모델**입니다.
쉽게 말해, 여러 정보를 보고 **결과가 얼마나 커질지 또는 작아질지**를 추정하는 방법입니다.

예를 들어 아래처럼 생각할 수 있습니다.

- 공부 시간으로 시험 점수 예측
- 집 크기와 역과의 거리로 집값 예측
- 광고비로 매출 예측

이런 문제의 공통점은 결과가 **숫자**라는 점입니다. 그래서 이를 회귀(regression) 문제라고 부릅니다.

---

### 1. 분류와 회귀의 차이

머신러닝 문제는 크게 두 가지로 많이 나뉩니다.

#### 회귀
- 결과가 숫자입니다.
- 예: 82점, 3억 5천만 원, 매출 1200만 원

#### 분류
- 결과가 범주입니다.
- 예: 합격/불합격, 정상/불량, 스팸/정상 메일

현재 노트북의 `TARGET`은 0과 1이라서 **엄밀히는 분류 문제**입니다.
다만 회귀 모델을 먼저 보면, **입력 변수와 출력 사이의 관계를 수식으로 어떻게 표현하는지**를 배우기에 좋습니다.

---

### 2. 회귀 모델의 핵심 아이디어

회귀 모델은 기본적으로 아래 질문에 답하려고 합니다.

**"입력값이 바뀌면 결과값이 얼마나 바뀌는가?"**

가장 기본적인 선형회귀(linear regression)는 이 관계를 직선 또는 평면처럼 단순한 형태로 표현합니다.

$$
y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p + \epsilon
$$

여기서 의미는 다음과 같습니다.

- $y$: 예측하려는 값
- $x_1, x_2, \dots$: 입력 변수
- $\beta_0$: 절편
- $\beta_1, \beta_2, \dots$: 각 변수의 영향력
- $\epsilon$: 설명하지 못한 오차

직관적으로 보면, **각 입력 변수에 점수표처럼 가중치를 붙여 더한 값으로 결과를 설명하는 방식**입니다.

---

### 3. 가장 쉬운 예시: 공부 시간과 시험 점수

예를 들어 아래 회귀식이 있다고 해봅시다.

$$
\text{시험점수} = 40 + 5 \times \text{공부시간}
$$

이 식은 이렇게 읽습니다.

- 공부를 전혀 안 해도 기본 점수는 40점
- 공부 시간이 1시간 늘면 점수는 평균적으로 5점 증가

즉, 여기서 `5`는 **공부시간의 회귀계수(coefficient)** 입니다.

예를 들어
- 2시간 공부하면 $40 + 5 \times 2 = 50$
- 6시간 공부하면 $40 + 5 \times 6 = 70$

이처럼 회귀 모델은 **변수 변화량이 결과에 주는 평균적 영향**을 숫자로 보여줍니다.

---

### 4. 선형회귀에서 "선형"이란 무슨 뜻인가?

많이 헷갈리는 부분인데, 여기서 선형은 **입력 변수를 계수와 곱해서 더하는 구조**라는 뜻입니다.

즉 아래 형태면 선형회귀입니다.

$$
y = 3 + 2x_1 - 0.7x_2
$$

변수가 여러 개 있어도 괜찮습니다. 직선이 아니라 다차원 공간의 평면이라고 생각하면 됩니다.

---

### 5. OLS는 무엇인가?

현재 노트북에서 사용한 `sm.OLS(...)`의 `OLS`는 **Ordinary Least Squares**, 즉 **최소제곱법**입니다.

이름은 어려워 보여도 아이디어는 단순합니다.

모델이 예측한 값과 실제 값의 차이(오차)를 구하고,
그 오차들의 제곱합이 가장 작아지도록 직선을 찾는 방법입니다.

$$
\text{오차} = \text{실제값} - \text{예측값}
$$

$$
\text{목표} = \sum (\text{오차})^2 \text{ 를 최소화}
$$

왜 제곱을 하느냐면
- 큰 오차에 더 큰 벌점을 주고
- 음수와 양수가 상쇄되지 않게 하기 위해서입니다.

즉 OLS는 **데이터에 가장 잘 맞는 선을 찾는 공식적인 방법**입니다.

---

### 6. 회귀 모델을 쓰는 이유

회귀 모델은 단순히 예측만 하는 도구가 아닙니다. 아래 두 목적에 모두 많이 씁니다.

#### 예측
- 내일 매출은 얼마일까?
- 아파트 가격은 얼마일까?

#### 해석
- 어떤 변수가 결과에 큰 영향을 주는가?
- 영향 방향은 양수인가 음수인가?
- 그 영향은 통계적으로 믿을 만한가?

특히 `statsmodels`는 **예측용**보다는 **해석용**으로 많이 씁니다.
그래서 `result.summary()`처럼 계수와 p-value를 자세히 보여줍니다.

---

### 7. 회귀계수를 직관적으로 읽는 방법

회귀계수는 아래처럼 해석합니다.

- 양수 계수: 그 변수가 커질수록 예측값이 커지는 경향
- 음수 계수: 그 변수가 커질수록 예측값이 작아지는 경향
- 절댓값이 클수록 영향력이 상대적으로 큼

예를 들어 집값 모델에서
- `평수 coef = 300`이면 평수가 늘수록 집값이 증가
- `역거리 coef = -200`이면 역에서 멀수록 집값이 감소

현재 노트북처럼 입력값을 표준화한 경우에는
**원래 단위 1 증가**가 아니라 **1 표준편차 증가**로 해석하는 편이 더 정확합니다.

---

### 8. 회귀 모델의 한계도 같이 알아야 한다

회귀 모델은 매우 강력하지만, 아무 상황에서나 무조건 맞는 것은 아닙니다.

#### 관계가 꼭 직선은 아닐 수 있다
- 실제 세상은 곡선 관계가 많습니다.
- 공부시간이 늘수록 점수가 계속 같은 비율로 오르지는 않을 수 있습니다.

#### 변수끼리 너무 비슷할 수 있다
- 예: `radius`, `perimeter`, `area`처럼 서로 강하게 연결된 변수들
- 이런 경우 계수 해석이 불안정해질 수 있습니다.

#### 이상치에 민감할 수 있다
- 일부 매우 큰 값이 직선을 많이 흔들 수 있습니다.

#### 분류 문제에는 적합하지 않을 수 있다
- 현재 `TARGET`은 0/1 입니다.
- 이런 경우는 OLS보다 로지스틱 회귀가 더 자연스럽습니다.

즉 회귀 모델은 **좋은 출발점**이지만, 문제 유형에 맞게 모델을 골라야 합니다.

---

### 9. 현재 노트북과 연결해서 이해하기

현재 노트북 흐름은 다음처럼 이해하면 됩니다.

1. 여러 유방암 관련 측정값을 입력 변수로 사용
2. 각 변수들을 표준화해서 단위를 맞춤
3. OLS를 사용해 `TARGET`과 입력 변수 간 관계를 먼저 살펴봄
4. `result.summary()`로 어떤 변수가 어떤 방향으로 관련되는지 확인

다만 여기서 중요한 점은,
**이 노트북의 OLS는 분류 문제를 설명하기 위한 학습용 관찰 도구에 가깝다**는 것입니다.

실제로 0/1을 예측하는 정식 분류 모델로는 아래의 로지스틱 회귀가 더 적합합니다.

$$
P(y=1 \mid x) = \frac{1}{1 + e^{-z}}
$$

여기서 $z$는 선형결합입니다.

$$
z = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p
$$

즉 로지스틱 회귀는 선형회귀의 구조를 가져오되, 출력을 확률 형태로 바꾼 모델이라고 볼 수 있습니다.

---

### 10. 한 줄 요약

회귀 모델은 **입력 변수와 결과값 사이의 관계를 수식으로 표현하고, 각 변수의 영향력을 숫자로 해석하는 모델**입니다.
현재 노트북에서는 OLS를 통해 변수와 타깃 사이의 관계를 먼저 살펴보고, 그 다음 `result.summary()`를 통해 그 관계를 통계적으로 읽는 연습을 하게 됩니다.

## `result.summary()`를 읽는 법: 기초부터 직관적으로 이해하기

`statsmodels`의 `result.summary()`는 **"이 회귀모델이 데이터를 얼마나 잘 설명하는지"**, 그리고 **"각 변수의 영향력이 통계적으로 믿을 만한지"**를 한 장의 표로 요약한 것입니다.

쉽게 말하면 이 표는 다음 3가지를 한 번에 보여줍니다.

1. **모델 전체 성적표**: 이 모델이 전체적으로 괜찮은가?
2. **변수별 성적표**: 어떤 변수가 많이, 어떤 방향으로 영향을 주는가?
3. **오류 진단표**: 결과를 얼마나 믿어도 되는가?

---

### 1. 먼저 큰 그림부터: 회귀식이 무엇인가?

OLS(최소제곱법) 선형회귀는 아래 형태를 가정합니다.

$$
y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p + \epsilon
$$

- $y$: 예측하려는 값
- $x_1, x_2, \dots$: 입력 변수들
- $\beta_0$: 절편(기본값)
- $\beta_1, \beta_2, \dots$: 각 변수의 영향력
- $\epsilon$: 모델이 설명하지 못한 오차

직관적으로 보면, **각 변수에 가중치(coefficients)를 곱해서 더한 값으로 결과를 설명하려는 모델**입니다.

예를 들어 집값 예측이라면 다음처럼 생각할 수 있습니다.

$$
집값 = 5000 + 300 \times \text{평수} - 200 \times \text{역거리}
$$

- 평수가 1 증가하면 집값이 300 증가
- 역거리가 1 증가하면 집값이 200 감소

즉 `summary()`는 이 식의 각 숫자가 얼마나 의미 있는지 검사한 결과표입니다.

---

### 2. 상단 요약: 모델 전체를 보는 영역

`summary()` 상단에는 보통 아래 항목들이 나옵니다.

#### `Dep. Variable`
- 종속변수, 즉 우리가 맞히려는 값입니다.
- 이 노트북에서는 `TARGET`입니다.

#### `Model`
- 어떤 회귀모델을 썼는지 보여줍니다.
- 현재 셀은 `OLS`입니다.

#### `No. Observations`
- 학습에 사용한 데이터 개수입니다.
- 데이터가 너무 적으면 결과가 불안정할 수 있습니다.

#### `Df Residuals`, `Df Model`
- 자유도(degrees of freedom)입니다.
- 엄밀한 통계 개념이지만, 초보자 관점에서는 **"데이터 수 대비 모델이 얼마나 복잡한가"** 정도로 이해하면 충분합니다.
- 변수가 많아질수록 모델은 더 유연해지지만, 과적합 위험도 커집니다.

#### `R-squared`
- 모델이 종속변수의 변동을 얼마나 설명하는지 나타냅니다.
- 값 범위는 보통 0에서 1 사이입니다.
- 예: `R-squared = 0.80`이면, **데이터 변화의 80%를 모델이 설명한다**는 뜻입니다.

직관적으로는 시험 점수 100점 만점 같은 느낌입니다.
- 1에 가까울수록 설명력이 높음
- 0에 가까울수록 설명력이 낮음

하지만 **높다고 해서 무조건 좋은 모델은 아닙니다.** 변수를 많이 넣으면 억지로 높아질 수도 있습니다.

#### `Adj. R-squared`
- 변수를 많이 넣는다고 무조건 점수가 올라가지 않도록 보정한 값입니다.
- 그래서 실제로는 `R-squared`보다 `Adj. R-squared`를 더 신뢰하는 경우가 많습니다.

예를 들어 의미 없는 변수를 20개 추가하면 `R-squared`는 조금 오를 수 있지만, `Adj. R-squared`는 오히려 내려갈 수 있습니다.

#### `F-statistic`, `Prob (F-statistic)`
- 모델 전체가 유의미한지 보는 검정입니다.
- 귀무가설: **"모든 계수가 0이다"**, 즉 이 모델은 아무 설명력도 없다.
- `Prob (F-statistic)`가 아주 작으면, 적어도 하나 이상의 변수는 의미가 있다고 봅니다.

보통 `p < 0.05`이면 통계적으로 유의하다고 해석합니다.

---

### 3. 핵심 표: 각 변수별 해석

가장 중요한 부분은 계수 테이블입니다. 보통 다음 열(column)들이 있습니다.

#### `coef`
- 각 변수의 회귀계수입니다.
- **해당 변수가 1 단위 증가할 때, 예측값이 얼마나 바뀌는지**를 의미합니다.

예를 들어 `coef = 2.5`라면:
- 다른 조건이 같을 때
- 그 변수 값이 1 증가하면
- 예측값이 평균적으로 2.5 증가

예를 들어 `mean radius`의 `coef = -0.12`라면,
- 다른 변수가 같다고 가정할 때
- `mean radius`가 1 증가할수록
- 예측값은 평균적으로 0.12 감소하는 방향이라는 뜻입니다.

#### 이 노트북에서는 왜 해석이 조금 다를까?
현재 코드는 `StandardScaler`를 사용했습니다. 즉 각 변수가 평균 0, 표준편차 1이 되도록 표준화됐습니다.

그래서 여기서 `coef = 0.8`은 **원래 단위 1 증가**가 아니라,
**"그 변수가 1 표준편차 증가할 때 예측값이 0.8만큼 바뀐다"**로 이해하는 게 더 정확합니다.

이 점 때문에 표준화된 회귀계수는 **변수 간 상대적 영향력 비교**에 유리합니다.

#### `std err`
- 계수 추정치의 표준오차입니다.
- 쉽게 말하면 **계수를 얼마나 흔들리게 추정했는지**를 나타냅니다.
- 작을수록 더 안정적입니다.

예를 들어
- `coef = 1.2`, `std err = 0.1`이면 꽤 안정적
- `coef = 1.2`, `std err = 2.0`이면 추정이 많이 흔들림

#### `t`
- 계수를 표준오차로 나눈 값입니다.
- 공식은 대략 다음과 같습니다.

$$
t = \frac{\text{coef}}{\text{std err}}
$$

- 절댓값이 클수록 **0이 아닐 가능성**이 커집니다.
- 즉 영향력이 있다고 볼 근거가 강해집니다.

#### `P>|t|`
- 해당 계수가 우연히 나온 값인지 판단하는 p-value입니다.
- 보통 `0.05`보다 작으면 유의하다고 해석합니다.

예를 들어
- `p = 0.001`이면 매우 유의미
- `p = 0.45`이면 우연일 가능성이 커서 해석에 신중해야 함

초보자 기준으로는 이렇게 기억하면 됩니다.
- **coef는 영향 방향과 크기**
- **p-value는 그 영향이 믿을 만한지**

#### `[0.025, 0.975]`
- 95% 신뢰구간입니다.
- 계수의 진짜 값이 이 구간 안에 있을 가능성이 높다는 뜻입니다.

예를 들어 신뢰구간이 `[0.2, 1.4]`이면 양수만 포함하므로, 양의 영향일 가능성이 큽니다.
반대로 `[-0.5, 0.8]`처럼 0을 포함하면, 영향이 없을 가능성도 있습니다.

---

### 4. 아래 진단값: 결과를 믿어도 되는가?

#### `Omnibus`, `Prob(Omnibus)`, `Jarque-Bera (JB)`
- 잔차(residual, 오차)가 정규분포에 가까운지 보는 지표입니다.
- 선형회귀의 이상적인 가정 중 하나가 오차의 정규성입니다.
- 다만 초급 단계에서는 **"오차가 너무 찌그러져 있지 않은가"** 정도로 받아들이면 충분합니다.

#### `Skew`
- 잔차 분포의 비대칭 정도입니다.
- 0에 가까우면 좌우대칭에 가깝습니다.

#### `Kurtosis`
- 분포가 얼마나 뾰족하거나 두꺼운 꼬리를 갖는지 보여줍니다.
- 정규분포의 기준값은 대략 3입니다.

#### `Durbin-Watson`
- 잔차의 자기상관을 점검합니다.
- 시계열 데이터에서 특히 중요합니다.
- 대략 2에 가까우면 큰 문제 없다고 봅니다.

#### `Cond. No.`
- 조건수(condition number)입니다.
- 너무 크면 변수들끼리 강하게 겹치는 다중공선성 문제가 있을 수 있습니다.
- 쉽게 말하면 **변수들이 서로 너무 비슷해서 모델이 누구 공인지 헷갈리는 상태**입니다.

예를 들어
- `mean radius`, `mean perimeter`, `mean area`는 서로 매우 강하게 연관될 수 있습니다.
- 그러면 계수는 불안정해지고 해석도 흔들립니다.

---

### 5. 이 노트북 결과를 읽을 때 꼭 알아야 할 점

현재 노트북의 `TARGET`은 0 또는 1입니다. 즉 **분류 문제**입니다.
그런데 `OLS`는 원래 연속형 숫자를 예측할 때 더 자연스러운 모델입니다.

그래서 이 OLS 결과는
- 선형관계를 빠르게 살펴보는 참고용으로는 쓸 수 있지만
- 확률적 분류 모델로 해석하기에는 한계가 있습니다.

특히 아래 점을 기억하면 좋습니다.

1. `coef` 부호
- 양수면 `TARGET=1` 쪽으로 가는 경향
- 음수면 `TARGET=0` 쪽으로 가는 경향

2. `coef` 크기
- 표준화된 상태이므로, 절댓값이 클수록 상대적으로 영향이 큰 변수일 가능성

3. `p-value`
- 작은 값이면 그 변수가 통계적으로 의미 있을 가능성이 큼

4. 하지만 분류 문제의 정식 해석은 `Logit` 또는 `LogisticRegression`이 더 적절
- 확률을 직접 모델링하기 때문입니다.

---

### 6. 아주 직관적인 비유

회귀 요약표를 **회사 성과 보고서**라고 생각하면 쉽습니다.

- `R-squared`: 회사 전체 성과
- `coef`: 직원 한 명이 성과에 미치는 영향
- `std err`: 그 평가가 얼마나 들쑥날쑥한지
- `t`, `P>|t|`: 그 직원의 공헌이 진짜인지, 우연인지
- `Cond. No.`: 직원들 역할이 너무 겹쳐서 평가가 꼬이지는 않는지

즉 `summary()`는 단순히 숫자표가 아니라,
**"모델 전체가 믿을 만한가?"**, **"어떤 변수가 중요해 보이는가?"**, **"그 해석은 안전한가?"**를 동시에 판단하는 보고서입니다.

---

### 7. 실전에서 가장 먼저 보는 순서

처음에는 아래 순서로 보면 됩니다.

1. `R-squared`, `Adj. R-squared`를 본다.
2. `Prob (F-statistic)`으로 모델 전체 유의성을 본다.
3. 관심 변수의 `coef` 부호와 크기를 본다.
4. 그 변수의 `P>|t|`가 0.05보다 작은지 본다.
5. `Cond. No.`가 너무 큰지 확인한다.

이 순서만 익혀도 `summary()` 표가 훨씬 덜 복잡하게 보입니다.

---

### 8. 한 줄 정리

`result.summary()`는 **모델의 전체 성능, 변수별 영향력, 해석의 신뢰도**를 한 번에 보여주는 통계 요약표입니다.
현재처럼 0/1 분류 문제에서는 참고용으로 볼 수 있지만, 최종 분류 해석은 로지스틱 회귀 결과와 함께 보는 것이 더 적절합니다.

## 데이터 전처리와 분할: 목적 및 자주 사용하는 추가 방법들

이 과정은 모델이 데이터에서 패턴을 원활하게 학습할 수 있도록 **"안내 길을 닦는 과정"**입니다. 특히 선형/로지스틱 회귀처럼 각 피처(Feature)간 단위를 맞추는 것과, y절편(상수)을 명시해야 하는 통계 라이브러리(`statsmodels`)를 쓸 때 필수적인 구조입니다.

### 실무에서 주로 사용하는 전처리 응용 방법 💡

1. **상황에 따른 스케일러(Scaler) 교체**
   위 코드에서는 평균 0, 분산 1을 만드는 `StandardScaler`를 사용했지만, 데이터 특성에 따라 다른 방법을 씁니다.
   - **`MinMaxScaler`**: 모든 데이터 값을 0과 1 사이로 압축합니다. 이미지 픽셀 데이터나 데이터의 최소/최대 범위가 절대적으로 정해져 있을 때 유용합니다.
   - **`RobustScaler`**: 중앙값(median)과 IQR(사분위값)을 사용하여 스케일링합니다. 데이터에 말도 안 되게 크거나 작은 '이상치(Outlier)'가 많을 때 값의 왜곡을 방지하기 위해 사용합니다.

2. **범주형 데이터의 원-핫 인코딩 (One-Hot Encoding)**
   알고리즘은 "서울", "부산", "남", "여" 같은 문자열을 바로 계산할 수 없습니다. 따라서 만약 문자형 범주 데이터가 있다면 스케일링 이전에 `pd.get_dummies()` 나 `OneHotEncoder`를 이용해서 0과 1로 된 희소 행렬로 바꾸어 주어야 합니다.

3. **데이터 누수(Data Leakage) 차단 - Scikit-learn \`Pipeline\` 사용**
   테스트 데이터나 검증 데이터에 학습 데이터의 정보가 스며드는 '데이터 누수'를 완벽히 막기 위해, 실무에서는 스케일러 모델과 머신러닝 모델을 하나로 묶는 `Pipeline` 객체를 자주 사용합니다. 
   ```python
   from sklearn.pipeline import make_pipeline
   from sklearn.linear_model import LogisticRegression

   # 파이프라인 형성 (스케일링 후 모델 적합이 한 묶음으로 처리됨)
   model_pipeline = make_pipeline(StandardScaler(), LogisticRegression())
   model_pipeline.fit(train_x, train_y)
   ```

4. **검증 데이터를 나누기 전(`train_test_split`)과 후의 순서 변경**
   엄밀히 말하면 **검증 세트를 먼저 분할(`train_test_split`)한 뒤, 검증 세트를 보지 않고 학습 데이터(Training set)로만 `StandardScaler.fit()`을 진행**하는 것이 정석입니다.
   위의 코드 흐름도 괜찮지만, 아주 미세한 데이터 누수(Validation의 정보가 StandardScaler에 반영되는 것)까지 방지하려면 `train_test_split`을 스케일링보다 앞서 수행하는 방식을 많이 씁니다.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

train_x_model = train_x.drop(columns=['const'])
val_x_model = val_x.drop(columns=['const'])

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(train_x_model, train_y)

val_pred = model.predict(val_x_model)
print('validation accuracy:', accuracy_score(val_y, val_pred))

validation accuracy: 0.9340659340659341


In [16]:
from sklearn.metrics import mean_absolute_error
pred_val = result.predict(val_x)

# MAE
mae = mean_absolute_error(val_y, pred_val)
print("Mean Absolute Error: ", mae)

Mean Absolute Error:  0.2188756414871274


## 회귀 모델 평가: MAE(Mean Absolute Error) 이해하기

모델이 얼마나 잘 예측했는지 측정하려면 **평가 지표(evaluation metric)**가 필요합니다.
회귀 모델에서 가장 직관적이고 많이 쓰이는 지표 중 하나가 **MAE(Mean Absolute Error)**입니다.

---

### 1. MAE가 무엇인가?

MAE는 **모델의 예측값과 실제값의 차이(오차)의 절댓값**을 평균한 것입니다.

$$
\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|
$$

여기서 의미는 다음과 같습니다.

- $y_i$: $i$번째 실제값
- $\hat{y}_i$: $i$번째 예측값
- $|y_i - \hat{y}_i|$: 오차의 절댓값
- $n$: 데이터 개수

**쉽게 말하면 "모델이 평균적으로 얼마나 빗나가는가"**를 나타냅니다.

---

### 2. 직관적인 예시

시험 점수를 예측하는 모델이 다음처럼 작동한다고 해봅시다.

| 학생 | 실제 점수 | 예측 점수 | 오차 | 절댓값 |
|------|---------|---------|-----|--------|
| A | 80 | 75 | -5 | 5 |
| B | 90 | 95 | 5 | 5 |
| C | 70 | 72 | 2 | 2 |
| D | 85 | 81 | -4 | 4 |

이 경우 MAE는

$$
\text{MAE} = \frac{5 + 5 + 2 + 4}{4} = \frac{16}{4} = 4
$$

즉 **평균적으로 4점씩 빗나간다**는 뜻입니다.

---

### 3. MAE의 특징

#### 직관적이다
- MAE의 단위는 원래 데이터의 단위와 같습니다.
- 예: 시험 점수 예측 모델이면 MAE도 "점수"
- 집값 예측 모델이면 MAE도 "원(currency)"

#### 모든 오차를 동등하게 취급한다
- 절댓값을 취하므로 작은 오차나 큰 오차나 동등하게 봅니다.
- 반대로 MSE(평균제곱오차)는 큰 오차에 더 큰 벌칙을 줍니다.

| 오차 | MAE에서의 기여 | MSE에서의 기여 |
|------|-------------|-------------|
| 1 | 1 | 1 |
| 2 | 2 | 4 |
| 3 | 3 | 9 |
| 10 | 10 | 100 |

보다시피 큰 오차(10)가 나면 MSE는 100으로 매우 커지지만, MAE는 10입니다.

#### 해석이 쉽다
- "평균적으로 X만큼 차이난다"로 바로 이해 가능
- RMSE(제곱근 평균제곱오차)보다 직관적

---

### 4. 현재 노트북의 예시에서 이해하기

현재 노트북은 유방암 진단 데이터로, `TARGET`은 0(악성) 또는 1(양성)입니다.

회귀 모델(OLS)이 예측한 값은 연속형 숫자입니다. 예를 들어:

| 실제 | 예측 | 오차 | 절댓값 |
|------|------|------|--------|
| 1 | 0.95 | 0.05 | 0.05 |
| 0 | 0.12 | -0.12 | 0.12 |
| 1 | 1.08 | 0.08 | 0.08 |
| 0 | -0.05 | -0.05 | 0.05 |

이 경우 MAE는

$$
\text{MAE} = \frac{0.05 + 0.12 + 0.08 + 0.05}{4} = 0.075
$$

즉 **평균적으로 0.075 정도의 오차**가 난다는 뜻입니다.

---

### 5. MAE로 모델 성능 평가하기

MAE는 작을수록 좋습니다.

| MAE | 해석 |
|-----|------|
| 0 | 완벽한 예측 (불가능) |
| < 0.1 | 매우 좋은 예측 (0/1 분류의 경우) |
| 0.1 ~ 0.3 | 좋은 예측 |
| 0.3 ~ 0.5 | 보통 예측 |
| > 0.5 | 나쁜 예측 |

현재 노트북이 계산한 MAE가 얼마나 작은지에 따라 모델이 좋은지 판단할 수 있습니다.

---

### 6. MAE vs RMSE vs R²

회귀 모델을 평가하는 지표는 여러 개입니다. 각각의 특징은:

| 지표 | 공식 | 특징 | 언제 쓰나 |
|------|------|------|---------|
| MAE | $\frac{1}{n}\sum\|y - \hat{y}\|$ | 직관적, 작은 오차 선호 | 모든 오차가 중요할 때 |
| MSE | $\frac{1}{n}\sum(y - \hat{y})^2$ | 큰 오차에 벌칙 | 이상치(outlier) 민감 |
| RMSE | $\sqrt{MSE}$ | MSE의 제곱근, 원래 단위로 돌림 | 가장 흔히 사용 |
| R² | $1 - \frac{SSres}{SStot}$ | 0~1 범위, 1에 가까울수록 좋음 | "얼마나 설명하나" 파악할 때 |

---

### 7. 현재 노트북의 맥락에서 MAE 사용의 한계

중요한 점: **현재 모델은 0/1 분류 문제를 회귀 모델로 해결하려고 시도**합니다.

그래서 MAE도 쓸 수 있지만, 아래 지표들이 더 적합합니다:

**분류 문제 평가 지표:**
- `Accuracy`: 전체 중 맞춘 비율
- `Precision`: 1이라고 예측한 것 중 실제 1인 비율
- `Recall`: 실제 1 중에 1이라고 예측한 비율
- `F1-score`: Precision과 Recall의 조화평균

이 지표들은 LogisticRegression처럼 확률 기반 분류 모델에서 더 의미 있습니다.

---

### 8. 한 줄 요약

**MAE는 "모델이 평균적으로 얼마나 빗나가는가"를 보여주는 가장 직관적인 평가 지표**입니다.

값이 작을수록 좋은 모델이고, 단위가 원래 데이터 단위와 같아서 이해하기 쉽습니다.

다만 현재처럼 0/1 분류 문제에서는 정확도(Accuracy)나 F1-score처럼 분류 특화 지표를 함께 봐야 더 완전한 평가를 할 수 있습니다.